# Cargar_datos · Avance #1 del PI (V1.0.1)

**Proyecto:** modelo de riesgo crediticio · **Rama de trabajo:** `developer`

En un proceso real este notebook casi nunca existe: la data ya vive en el DWH o en el Datalake de la empresa como resultado de otro proceso. Aquí usamos un dataset **no productivo** de ejemplo, entregado como Excel, y lo dejamos en `Base_de_datos.csv` en la raíz del repo, que es el archivo que exige la estructura del proyecto.

Flujo: ubicar la raíz → inspeccionar el Excel → convertir a CSV y verificar → cargar el CSV.

In [ ]:
# 1. Ubicar la raíz del repo
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "mlops_pipeline").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
assert (RAIZ / "mlops_pipeline").is_dir(), "No encontré mlops_pipeline/: abre el notebook dentro del repo"

RUTA_XLSX = RAIZ / "Base_de_datos.xlsx"
RUTA_CSV = RAIZ / "Base_de_datos.csv"
print("Raíz del proyecto:", RAIZ)
print("¿Existe el xlsx?:", RUTA_XLSX.exists(), "| ¿Existe el CSV?:", RUTA_CSV.exists())

In [ ]:
# 2. Inspeccionar el Excel original (solo lectura)
if RUTA_XLSX.exists():
    with pd.ExcelFile(RUTA_XLSX) as libro:
        print("Hojas del archivo:", libro.sheet_names)
        df_xlsx = pd.read_excel(libro, sheet_name="Hoja1")
    print(f"Hoja1 → Filas: {df_xlsx.shape[0]:,} | Columnas: {df_xlsx.shape[1]}")
else:
    df_xlsx = None
    print("No está el xlsx (normal si clonaste el repo: el .gitignore lo excluye). Se usa el CSV versionado.")

In [ ]:
# 3. Convertir a CSV y verificar que no se perdió nada
if df_xlsx is not None:
    df_xlsx.to_csv(RUTA_CSV, index=False, encoding="utf-8")
    copia = pd.read_csv(RUTA_CSV)
    assert copia.shape == df_xlsx.shape, "El CSV no tiene las mismas filas y columnas que la hoja"
    assert list(copia.columns) == list(df_xlsx.columns), "Cambiaron los nombres de las columnas"
    assert not any(c.startswith("Unnamed") for c in copia.columns), "Se coló el índice como columna"
    print(f"CSV escrito y verificado: {RUTA_CSV.name} ({RUTA_CSV.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# 4. Cargar la base que usará todo el proyecto
df = pd.read_csv(RUTA_CSV)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
# 5. Estructura: tipos y memoria
df.info()

In [ ]:
# 6. Primer vistazo de calidad
pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "valores_distintos": df.nunique(),
})

## Conclusiones de la carga

- La base tiene **10.763 filas y 23 columnas**; la conversión Excel → CSV no perdió filas ni columnas.
- `fecha_prestamo` llega como **texto**: hay que convertirla a fecha en el EDA.
- `tendencia_ingresos` es de texto pero tiene **46 valores distintos** cuando esperaríamos 3 categorías: algo raro hay ahí.
- Hay nulos en varias columnas (el más alto, `promedio_ingresos_datacredito`, ~27 %).

Todo esto lo trabajamos en `comprension_eda.ipynb`.